In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
log2_path = "../results/deseq2_rsem/hydra_all_deseq2_lg2_t_to_g.tsv"
pval_path = "../results/deseq2_rsem/hydra_all_deseq2_pvalues_t_to_g.tsv"

In [3]:
'''read_differential_expression_tables
    
    This function reads the comprehensive log2FoldChange and pvalue dataframes, in which all 
    transcriptomic data resides. It then transforms the ID column to a gene_id column for further
    merging steps. This will ensure same gene identifiers as strings across different dataframes.
    
    :param log2FoldChange_path
        :type str
    :param pValue_path
        :type str
        
    :returns diff_exp_table, p_value_table
        :type tupe(pd.DataFrame, pd.DataFrame)

'''
def read_differential_expression_tables(log2FoldChange_path:str,pValue_path:str)->tuple:
    try:
        
        print("[*] Reading dataframes ...")
        diff_exp_table = pd.read_csv(log2FoldChange_path, sep="\t")
        p_value_table = pd.read_csv(pValue_path, sep="\t")
        # renaming ID column for merging
        print("[*] Transforming ID column ...")
        p_value_table.rename(columns={"ID":"gene_id"}, inplace=True)
        diff_exp_table.rename(columns={"ID":"gene_id"}, inplace=True)
        # equalize gene_ids for merging
        diff_exp_table["gene_id"] = diff_exp_table["gene_id"].apply(lambda x: x.split("-")[1])
        p_value_table["gene_id"] = p_value_table["gene_id"].apply(lambda x: x.split("-")[1])
        
        print("[*] Transforming NA values to none-differentially expressed genes ...")
        # handling missing data as non differentially expressed genes
        diff_exp_table = diff_exp_table.fillna(0.0)
        p_value_table = p_value_table.fillna(1.0)
        
        print("[+] DONE")
        return diff_exp_table, p_value_table
    except Exception as e:
        raise Exception("[-] ERROR reading transcriptome tables with exception: {}".format(e)) 

In [4]:
log2_df, pval_df = read_differential_expression_tables(log2_path, pval_path)

[*] Reading dataframes ...
[*] Transforming ID column ...
[*] Transforming NA values to none-differentially expressed genes ...
[+] DONE


In [55]:
sliced_log2_df = log2_df[["gene_id","HydraRecolonization_Conventionalized_vs_GF","HydraRecolonization_Cvbct_vs_GF","HydraAHL_3OHC12_vs_control"]]
sliced_pval_df = pval_df[["gene_id","HydraRecolonization_Conventionalized_vs_GF","HydraRecolonization_Cvbct_vs_GF","HydraAHL_3OHC12_vs_control"]]

numeric_columns = sliced_pval_df.select_dtypes(include=[np.number])
filtering = numeric_columns.where(numeric_columns <= 0.05)
sliced_pval_df_filtered = sliced_pval_df.copy()
sliced_pval_df_filtered[["HydraRecolonization_Conventionalized_vs_GF","HydraRecolonization_Cvbct_vs_GF","HydraAHL_3OHC12_vs_control"]] = filtering

In [57]:
gene_list = sliced_pval_df_filtered.dropna()["gene_id"].to_list()

In [58]:
sliced_log2_df[sliced_log2_df.gene_id.isin(gene_list)].sort_values(by="HydraRecolonization_Conventionalized_vs_GF")

,gene_id,HydraRecolonization_Conventionalized_vs_GF,HydraRecolonization_Cvbct_vs_GF,HydraAHL_3OHC12_vs_control
5505,G021899,-4.712668,-3.561649,-2.847061
7457,G018899,-4.070329,-2.845071,-3.113894
9922,G010193,-4.062821,-3.028726,-2.283620
965,G003347,-4.051146,-2.305762,2.186237
622,G023470,-3.724197,-2.306208,2.803672
993,G006802,-3.154914,-2.473108,4.355255
7441,G018898,-3.008052,-2.970414,-3.107978
15714,G013093,-2.646851,-1.842228,-0.999587
2983,G018174,-2.483747,-0.924724,1.079544
5376,G018172,-2.358388,-1.014108,0.971127
